In [ ]:
# Preparación del dataset de matrícula y secciones

import pandas as pd





path = '02_data_formatting/Matricula_Secciones_Final.csv'

df = pd.read_csv(path)
#print(*df.columns, sep='\n')

df = df.drop(columns=['lactantes','deambulantes','s2','s3','s4','s5','_snu','v_lactantes','v_deambulantes',
'v_s2',
'v_s3',
'v_s4',
'v_s5',
'v_snu',
's_lactantes',
's_deambulantes',
's_s2',
's_s3',
's_s4',
's_s5',
'_seclactantes',
'_secdeambu',
'_secs2',
'_secs3',
'_secs4',
'_secs5',
'_sec1',
'_sec2',
'_sec3',
'_sec4',
'_sec5',
'_sec6',
'_sec7',
'_sec8',
'_sec9',
'_sec10',
'_sec11',
'_sec12',
'_sec1314',
'multi_ini',
'multi_pri',
'multi_sec',
'multinivel'])




nuevos_nombres = {
    'provincia': 'provincia',
    'departamento': 'departamento',
    'sector': 'sector',
    'ambito': 'ambito'
}

for i in range(1, 13):
    # Definimos el sufijo: del 1 al 5 es "grado", del 6 al 12 es "anio"
    if i <= 6:
        etiqueta = f'grado_{i}'
    else:
        # 6 es 1er anio, 7 es 2do anio, etc.
        etiqueta = f'anio_{i-6}'

    nuevos_nombres[f'_{i}'] = f'total_{etiqueta}'
    nuevos_nombres[f'v_{i}'] = f'varones_{etiqueta}'
    nuevos_nombres[f'r_{i}'] = f'repitentes_{etiqueta}'
    nuevos_nombres[f's_{i}'] = f'sobreedad_{etiqueta}'

# Casos especiales (13 y 14 suelen ser anios superiores)
nuevos_nombres.update({
    '_1314': 'total_anio_sup', 'v_1314': 'varones_anio_sup',
    'r_1314': 'repitentes_anio_sup', 's_1314': 'sobreedad_anio_sup'
})

df = df.rename(columns=nuevos_nombres)

# 2. Convertir a numérico (Indispensable para calcular)
# Seleccionamos todas las columnas excepto las 4 de texto iniciales
cols_a_convertir = df.columns.drop(['provincia', 'departamento', 'sector', 'ambito'])
for col in cols_a_convertir:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# 3. Cálculos de Mujeres, Tasas de Repitencia y Sobreedad
for i in range(1, 13):
    if i <= 6:
        e = f'grado_{i}'
    else:
        e = f'anio_{i-6}'

    # Crear columna de Mujeres
    df[f'mujeres_{e}'] = df[f'total_{e}'] - df[f'varones_{e}']

    # Tasa de Repitencia (Repitentes / Total * 100)
    # Usamos .replace(0, 1) en el denominador para evitar división por cero
    df[f'tasa_repitencia_{e}'] = (df[f'repitentes_{e}'] / df[f'total_{e}'].replace(0, 1)) * 100

    # Tasa de Sobreedad (Sobreedad / Total * 100)
    df[f'tasa_sobreedad_{e}'] = (df[f'sobreedad_{e}'] / df[f'total_{e}'].replace(0, 1)) * 100

# Verificamos los nuevos cálculos para el primer grado
#display(df[['total_grado_5', 'varones_grado_5', 'mujeres_grado_5', 'tasa_repitencia_grado_5']].head())

display(df.info())

# ==============================
# 💾 Guardar dataset matrícula limpio
# ==============================

import os

output_path = '03_data_cleaning'
os.makedirs(output_path, exist_ok=True)

df.to_excel(f'{output_path}/matricula_limpia.xlsx', index=False)

print("✅ Matrícula guardada correctamente")

In [ ]:
import pandas as pd
import unicodedata

path = '02_data_formatting/Establecimiento_Caracteristicas_Final.csv'

df = pd.read_csv(path)

# ==============================
# 🧹 LIMPIAR NOMBRES DE COLUMNAS
# ==============================

def limpiar_texto(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )

df.columns = [limpiar_texto(col) for col in df.columns]

# ==============================
# 🗑️ DROP
# ==============================

df = df.drop(columns=[
'EquipamientoEstablecimientoTelevisor',
'EquipamientoEstablecimientoSistemamultimediaoCanon',
'EquipamientoEstablecimientoScanner',
'EquipamientoEstablecimientoWebcam',
'EquipamientoEstablecimientoReproductordeCD',
'EquipamientoEstablecimientoReproductordeDVD',
'EquipamientoEstablecimientoImpresora',
'EquipamientoEstablecimientoEquipoemisorderadioAMFM',
'EquipamientoEstablecimientoEquiporeceptorderadioAMFM',
'EquipamientoEstablecimientoServidorparausoescolar',
'EquipamientoEstablecimientoImpresora3D',
'EquipamientoEstablecimientoEquipodeSonido',
'EquipamientoEstablecimientoPizarrasdigitales'
], errors='ignore')

# ==============================
# 🔍 VALIDACIÓN
# ==============================

print("Columnas finales:", len(df.columns))
display(df.info())

# ==============================
# 💾 GUARDADO
# ==============================

df.to_excel(f'{output_path}/caracteristicas_limpio.xlsx', index=False)

print("✅ Características guardado correctamente")


In [ ]:
import pandas as pd

path = '02_data_formatting/Trayectoria_Sexo_Final.csv'

# ==============================
# 📥 CARGA CORRECTA
# ==============================

df = pd.read_csv(path)  # <-- SIN sep=';'

# ==============================
# 🧹 DROP (robusto)
# ==============================

df = df.drop(columns=[
'entrados_1','entrados_2','entrados_3','entrados_4','entrados_5','entrados_6',
'entrados_7','entrados_8','entrados_9','entrados_10','entrados_11','entrados_12','entrados_1314',
'm_entrados_1','m_entrados_2','m_entrados_3','m_entrados_4','m_entrados_5','m_entrados_6',
'm_entrados_7','m_entrados_8','m_entrados_9','m_entrados_10','m_entrados_11','m_entrados_12','m_entrados_1314',
'scp_1','scp_2','scp_3','scp_4','scp_5','scp_6','scp_7','scp_8','scp_9','scp_10','scp_11','scp_12','scp_1314',
'm_scp_1','m_scp_2','m_scp_3','m_scp_4','m_scp_5','m_scp_6','m_scp_7','m_scp_8','m_scp_9','m_scp_10','m_scp_11','m_scp_12','m_scp_1314',
'regulares_1','regulares_2','regulares_3','regulares_4','regulares_5','regulares_6','regulares_7','regulares_8',
'regulares_9','regulares_10','regulares_11','regulares_12','regulares_1314',
'm_regulares_1','m_regulares_2','m_regulares_3','m_regulares_4','m_regulares_5','m_regulares_6',
'm_regulares_7','m_regulares_8','m_regulares_9','m_regulares_10','m_regulares_11','m_regulares_12','m_regulares_1314',
'otros_1','otros_2','otros_3','otros_4','otros_5','otros_6','otros_7','otros_8','otros_9','otros_10','otros_11','otros_12','otros_1314',
'm_otros_1','m_otros_2','m_otros_3','m_otros_4','m_otros_5','m_otros_6','m_otros_7','m_otros_8','m_otros_9','m_otros_10','m_otros_11','m_otros_12','m_otros_1314'
], errors='ignore')

# ==============================
# 🔄 RENOMBRE (lo tuyo está bien)
# ==============================

nuevos_nombres = {
    'provincia': 'provincia',
    'departamento': 'departamento',
    'sector': 'sector',
    'ambito': 'ambito',
    'primaria_egresados': 'egresados_primaria',
    'primaria_m_egresados': 'egresados_mujeres_primaria',
    'secundaria_egresados': 'egresados_secundaria',
    'secundaria_m_egresados': 'egresados_mujeres_secundaria'
}

# (mantengo tu lógica porque está BIEN 👌)
dic_prefijos = {
    'inicial_': 'inscritos_marzo_',
    'm_inicial_': 'inscritos_marzo_mujeres_',
    'ssp_': 'desercion_',
    'm_ssp_': 'desercion_mujeres_',
    'ultimo_': 'alumnos_diciembre_',
    'm_ultimo_': 'alumnos_diciembre_mujeres_',
    'promovidos_': 'aprobados_',
    'm_promovidos_': 'aprobados_mujeres_',
    'promovidos_ex_': 'aprobados_mesas_',
    'm_promovidos_ex_': 'aprobados_mesas_mujeres_',
    'nopromo_': 'reprobados_',
    'm_nopromo_': 'reprobados_mujeres_'
}

for i in range(1, 13):
    etiqueta = f'grado_{i}' if i <= 6 else f'anio_{i-6}'
    for original, intuitivo in dic_prefijos.items():
        nuevos_nombres[f'{original}{i}'] = f'{intuitivo}{etiqueta}'

for original, intuitivo in dic_prefijos.items():
    nuevos_nombres[f'{original}1314'] = f'{intuitivo}anio_superior'

df = df.rename(columns=nuevos_nombres)

# ==============================
# 🔢 NUMÉRICOS
# ==============================

cols_base = ['provincia', 'departamento', 'sector', 'ambito']
cols_numericas = df.columns.drop(cols_base)
df[cols_numericas] = df[cols_numericas].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# ==============================
# 💾 GUARDADO
# ==============================

df.to_excel(f'{output_path}/trayectoria_limpia.xlsx', index=False)

print("✅ Trayectoria guardada correctamente")